In [ ]:
# Cleaned & anonymized multivariate simulation + evaluation script
# NOTE: Update the sys.path.append(...) below to your local CausalICM repository root before running.

import sys
# <-- replace with your local CausalICM repo root (do NOT point to __pycache__)
sys.path.append("/path/to/your/CausalICM/")

import os
import random
import copy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# reproducibility
seed_value = 1234
os.environ['PYTHONHASHSEED'] = str(seed_value)
random.seed(seed_value)
np.random.seed(seed_value)

# CausalICM functions (assumes folder structure CausalICM/functions/)
from CausalICM.functions.data_simulation_multivariate import data_simulation1
from CausalICM.functions.model_training import ICM
from CausalICM.functions.weightedMSE import weighted_mean_squared_error

# ML & utils
import GPy
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import Ridge as ridge
from sklearn.ensemble import RandomForestRegressor as rfr

# R interface (if used downstream)
import rpy2.robjects as robjects
import rpy2.robjects.numpy2ri as np2ri
robjects.numpy2ri.activate()

# -------------------------------
# Helper functions / parameters
# -------------------------------
rmse = lambda x, y: np.power(mean_squared_error(x, y), 0.5)

linear_reg_parameter_pairs = [
    (ridge(), {'alpha': np.logspace(-7, 7, 8), 'tol': [1e-3]})
]

rfr_reg_parameter_pairs = [
    (rfr(), {'min_samples_leaf': list(range(1, 11, 3)), 'n_estimators': [100, 200]})
]

def get_best_model_from_gridcv(X, Y, reg_parameter_pairs, print_flag=False, verbose=0):
    """Return best estimator found by GridSearchCV (selection by validation RMSE)."""
    val_errs = []
    models = []
    x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.2)
    for reg, parameters in reg_parameter_pairs:
        if print_flag:
            print('Trying', str(reg)[:50])
        model = GridSearchCV(reg, parameters, cv=2, refit=True, verbose=verbose, n_jobs=10)
        model.fit(x_train, y_train)
        val_pred = model.predict(x_test)
        val_errs.append(rmse(y_test, val_pred))
        models.append(copy.deepcopy(model.best_estimator_))
    min_ind = val_errs.index(min(val_errs))
    if verbose:
        print(str(models[min_ind]), 'val rmse:{}'.format(val_errs[min_ind]))
    return copy.deepcopy(models[min_ind])

# -------------------------------
# Kallus-style GP two-step adjustment
# -------------------------------
def run_methodGP(X_rct, Y_rct, T_rct, X_obs, Y_obs, T_obs, X_eval):
    """
    Fit separate GPs on observational control/treatment and:
      - return linear model for eta estimated on RCT (residual),
      - omega predictions on observational and evaluation points.
    """
    assert X_rct.shape[0] == Y_rct.shape[0]
    assert T_rct.shape[0] == Y_rct.shape[0]
    assert X_obs.shape[0] == Y_obs.shape[0]
    assert T_obs.shape[0] == Y_obs.shape[0]

    input_dim = X_obs.shape[1]
    kern = GPy.kern.RBF(input_dim=input_dim, variance=1., lengthscale=1.)

    # Control GP
    f0_gp = GPy.models.GPRegression(np.vstack(X_obs[T_obs == 0]), np.vstack(Y_obs[T_obs == 0]), kern.copy())
    f0_gp.optimize()

    # Treated GP
    f1_gp = GPy.models.GPRegression(np.vstack(X_obs[T_obs == 1]), np.vstack(Y_obs[T_obs == 1]), kern.copy())
    f1_gp.optimize()

    # omega predictions
    omega_rct_pred = f1_gp.predict(np.vstack(X_rct))[0] - f0_gp.predict(np.vstack(X_rct))[0]
    omega_obs_pred = f1_gp.predict(np.vstack(X_obs))[0] - f0_gp.predict(np.vstack(X_obs))[0]
    omega_eval_pred = f1_gp.predict(np.vstack(X_eval))[0] - f0_gp.predict(np.vstack(X_eval))[0]

    # adjusted ITE on RCT (IPW style using RCT prop)
    prop_rct = np.mean(T_rct)

    def ite_adjusted_outcome(Y, T, _propensity, c=None):
        c = np.mean(Y) if c is None else c
        assert 0 < _propensity < 1
        _ite = np.zeros(Y.shape)
        _ite[T == 1] = (Y[T == 1] - c) / _propensity
        _ite[T == 0] = -(Y[T == 0] - c) / (1 - _propensity)
        return _ite

    cate_rct_est = ite_adjusted_outcome(Y_rct, T_rct, _propensity=prop_rct)
    eta_rct_est = np.vstack(cate_rct_est) - np.vstack(omega_rct_pred)

    # regress eta on X_rct (choose best linear reg via small grid)
    best_eta_model = get_best_model_from_gridcv(np.asarray(X_rct).reshape(len(X_rct), -1), eta_rct_est, linear_reg_parameter_pairs)
    return copy.deepcopy(best_eta_model), omega_obs_pred, omega_eval_pred

# -------------------------------
# Load best rho and initialize storage
# -------------------------------
best_rho = pd.read_csv('sim1_best_rho.csv').values[0][0]

n_iter = 100
test_data_1d = np.arange(-2.0, 2.04, 0.04)
# form a grid of 5 identical covariates for evaluation (shape: n_eval x 5)
test_data = np.column_stack([test_data_1d for _ in range(5)])

rmse_naiveGP_exp = np.zeros(n_iter)
rmse_naiveGP_obs = np.zeros(n_iter)
rmse_KallusGP = np.zeros(n_iter)
rmse_ICM = np.zeros(n_iter)

bias_naiveGP_exp = np.zeros(n_iter)
bias_naiveGP_obs = np.zeros(n_iter)
bias_KallusGP = np.zeros(n_iter)
bias_ICM = np.zeros(n_iter)

var_naiveGP_exp = np.zeros(n_iter)
var_naiveGP_obs = np.zeros(n_iter)
var_KallusGP = np.zeros(n_iter)
var_ICM = np.zeros(n_iter)

bias_naive_exp_df = pd.DataFrame(test_data)
bias_naive_obs_df = pd.DataFrame(test_data)
bias_kallus_df = pd.DataFrame(test_data)
bias_icm_df = pd.DataFrame(test_data)

all_data = []

# -------------------------------
# Main simulation loop (multivariate)
# -------------------------------
for i in range(n_iter):
    print(f"Iteration {i+1}/{n_iter}")
    data_exp, data_obs = data_simulation1(1000, beta0=-10.0, beta1=-8.0, beta2=-8.0)

    # construct dataframes
    df_exp = pd.DataFrame(data_exp, columns=["S", "X1", "X2", "X3", "X4", "X5", "A", "Y"])
    df_exp["datasetNo"] = i + 1
    df_obs = pd.DataFrame(data_obs, columns=["S", "X1", "X2", "X3", "X4", "X5", "A", "Y"])
    df_obs["datasetNo"] = i + 1
    combined_df = pd.concat([df_exp, df_obs], ignore_index=True)
    all_data.append(combined_df)

    # prepare train/test groups by treatment
    X0_obs = data_obs[data_obs[:, 6] == 0, 1:6]
    X1_obs = data_obs[data_obs[:, 6] == 1, 1:6]
    Y0_obs = data_obs[data_obs[:, 6] == 0, 7]
    Y1_obs = data_obs[data_obs[:, 6] == 1, 7]

    X0_exp = data_exp[data_exp[:, 6] == 0, 1:6]
    X1_exp = data_exp[data_exp[:, 6] == 1, 1:6]
    Y0_exp = data_exp[data_exp[:, 6] == 0, 7]
    Y1_exp = data_exp[data_exp[:, 6] == 1, 7]

    # True CATE (depends on your DGP; here we use first two covariates)
    true_cate = 1 + test_data[:, 0] + test_data[:, 1]

    # -------------------------------
    # Naive GP on observational data
    # -------------------------------
    kern = GPy.kern.RBF(input_dim=5, variance=1., lengthscale=1.)
    m0_obs = GPy.models.GPRegression(np.vstack(X0_obs), np.vstack(Y0_obs), kern.copy()); m0_obs.optimize()
    m1_obs = GPy.models.GPRegression(np.vstack(X1_obs), np.vstack(Y1_obs), kern.copy()); m1_obs.optimize()
    mu0_obs = m0_obs.predict(np.vstack(test_data))[0]
    mu1_obs = m1_obs.predict(np.vstack(test_data))[0]
    naiveCATE_obs = mu1_obs - mu0_obs

    # -------------------------------
    # Naive GP on experimental data
    # -------------------------------
    m0_exp = GPy.models.GPRegression(np.vstack(X0_exp), np.vstack(Y0_exp), kern.copy()); m0_exp.optimize()
    m1_exp = GPy.models.GPRegression(np.vstack(X1_exp), np.vstack(Y1_exp), kern.copy()); m1_exp.optimize()
    mu0_exp = m0_exp.predict(np.vstack(test_data))[0]
    mu1_exp = m1_exp.predict(np.vstack(test_data))[0]
    naiveCATE_exp = mu1_exp - mu0_exp

    # -------------------------------
    # Kallus-style GP two-step adjustment
    # -------------------------------
    eta_model, omega_obs_pred, omega_eval_pred = run_methodGP(
        data_exp[:, 1:6], data_exp[:, 7], data_exp[:, 6],
        data_obs[:, 1:6], data_obs[:, 7], data_obs[:, 6],
        test_data
    )
    tau_pred = eta_model.predict(test_data) + omega_eval_pred

    # -------------------------------
    # ICM predictions using best_rho
    # -------------------------------
    m0_icm, m1_icm = ICM(
        X_E = data_exp[:, 1:6], X_O = data_obs[:, 1:6],
        T_E = data_exp[:, 6], T_O = data_obs[:, 6],
        Y_E = data_exp[:, 7], Y_O = data_obs[:, 7],
        r = 2, ID = 5, AD = [0, 1, 2, 3, 4], rho = best_rho
    )

    # prepare evaluation inputs for predict_noiseless: X + T flag
    X_eval_control = np.c_[test_data, np.zeros((test_data.shape[0], 1))]
    X_eval_treated = np.c_[test_data, np.ones((test_data.shape[0], 1))]

    predY0_exp = m0_icm.predict_noiseless(X_eval_control)
    predY1_exp = m1_icm.predict_noiseless(X_eval_treated)
    predCATE_icm = predY1_exp[0] - predY0_exp[0]

    # -------------------------------
    # Compute metrics and store
    # -------------------------------
    rmse_naiveGP_exp[i] = mean_squared_error(naiveCATE_exp, true_cate, squared=False)
    rmse_naiveGP_obs[i] = mean_squared_error(naiveCATE_obs, true_cate, squared=False)
    rmse_KallusGP[i] = mean_squared_error(tau_pred, true_cate, squared=False)
    rmse_ICM[i] = mean_squared_error(predCATE_icm, true_cate, squared=False)

    bias_naiveGP_exp[i] = np.mean(naiveCATE_exp - true_cate)
    bias_naiveGP_obs[i] = np.mean(naiveCATE_obs - true_cate)
    bias_KallusGP[i] = np.mean(tau_pred - true_cate)
    bias_ICM[i] = np.mean(predCATE_icm - true_cate)

    var_naiveGP_exp[i] = np.var(naiveCATE_exp)
    var_naiveGP_obs[i] = np.var(naiveCATE_obs)
    var_KallusGP[i] = np.var(tau_pred)
    var_ICM[i] = np.var(predCATE_icm)

    bias_naive_exp_df[f"Iteration_{i+1}"] = naiveCATE_exp - true_cate
    bias_naive_obs_df[f"Iteration_{i+1}"] = naiveCATE_obs - true_cate
    bias_kallus_df[f"Iteration_{i+1}"] = tau_pred - true_cate
    bias_icm_df[f"Iteration_{i+1}"] = predCATE_icm - true_cate

# -------------------------------
# Save final results
# -------------------------------
final_df_sim1_mult = pd.concat(all_data, ignore_index=True)
final_df_sim1_mult.to_csv('datasets_sim1_mult.csv', index=False)

sim1_modelComparison_RMSEs_df = pd.DataFrame(
    np.c_[rmse_naiveGP_exp, rmse_naiveGP_obs, rmse_KallusGP, rmse_ICM,
          bias_naiveGP_exp, bias_naiveGP_obs, bias_KallusGP, bias_ICM,
          var_naiveGP_exp, var_naiveGP_obs, var_KallusGP, var_ICM],
    columns=['RMSE_naiveGPexp','RMSE_naiveGPobs','RMSE_KallusGP','RMSE_ICM',
             'bias_naiveGPexp','bias_naiveGPobs','bias_KallusGP','bias_ICM',
             'var_naiveGPexp','var_naiveGPobs','var_KallusGP','var_ICM']
)

sim1_modelComparison_RMSEs_df.to_csv('sim1_modelComparison.csv', index=False)
bias_naive_exp_df.to_csv('bias_naive_exp.csv', index=False)
bias_naive_obs_df.to_csv('bias_naive_obs.csv', index=False)
bias_kallus_df.to_csv('bias_kallus.csv', index=False)
bias_icm_df.to_csv('bias_icm.csv', index=False)
